# DQN Transformer Profiling

This notebook profiles the JAX/Flax transformer DQN update and action-sampling paths.

It reports:
- first-call (compile + execute) time
- steady-state per-step time
- optional architecture sweep
- optional JAX trace export

In [1]:
import os
import pathlib
import sys
import time
from pprint import pprint

# Must be set before importing JAX so the process only sees physical GPU 4.
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import jax
import jax.numpy as jnp

ROOT = pathlib.Path.cwd().resolve().parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

# Use the visible GPU by default; fall back to CPU if unavailable.
PROFILE_DEVICE = jax.devices("gpu")[0] if jax.devices("gpu") else jax.devices("cpu")[0]
print("Profiling device:", PROFILE_DEVICE)

CUDA_VISIBLE_DEVICES: 4
JAX version: 0.6.1
Devices: [CudaDevice(id=0)]
Profiling device: cuda:0


In [2]:
from impls.agents.dqn_transformer import GCDQNTransformerAgent, get_config

NUM_GRID_STATES = 12
ACTION_DIM = 6
GRID_SIZE = 5
N_TOKENS = GRID_SIZE * GRID_SIZE
BATCH_SIZE = 64

def block_tree(tree):
    jax.tree_util.tree_map(
        lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
        tree,
    )

def make_one_hot_flat(key, batch_size=BATCH_SIZE, n_tokens=N_TOKENS, num_states=NUM_GRID_STATES):
    cell_ids = jax.random.randint(key, (batch_size, n_tokens), 0, num_states)
    return jax.nn.one_hot(cell_ids, num_states, dtype=jnp.float32).reshape(batch_size, -1)

def make_batch(key, batch_size=BATCH_SIZE):
    k1, k2, k3, k4, k5 = jax.random.split(key, 5)
    observations = make_one_hot_flat(k1, batch_size=batch_size)
    next_observations = make_one_hot_flat(k2, batch_size=batch_size)
    goals = make_one_hot_flat(k3, batch_size=batch_size)
    actions = jax.random.randint(k4, (batch_size,), 0, ACTION_DIM, dtype=jnp.int32)
    rewards = jax.random.uniform(k5, (batch_size,), minval=0.0, maxval=1.0)
    masks = jnp.ones((batch_size,), dtype=jnp.float32)
    return {
        "observations": observations,
        "next_observations": next_observations,
        "actions": actions,
        "rewards": rewards,
        "masks": masks,
        "value_goals": goals,
        "actor_goals": goals,
    }

In [3]:
cfg = get_config()

# Baseline profile config (edit these as needed).
cfg.token_input_representation = "one_hot_flat"
cfg.transformer_use_goals = False
cfg.transformer_d_model = 128
cfg.transformer_num_layers = 1
cfg.transformer_num_heads = 4
cfg.transformer_mlp_dim = 256
cfg.critic_ensemble_size = 1
cfg.batch_size = BATCH_SIZE

with jax.default_device(PROFILE_DEVICE):
    key = jax.random.PRNGKey(0)
    ex_obs = make_one_hot_flat(key, batch_size=1)
    ex_actions = jnp.array([ACTION_DIM - 1], dtype=jnp.int32)

    agent = GCDQNTransformerAgent.create(
        seed=0,
        ex_observations=ex_obs,
        ex_actions=ex_actions,
        config=cfg,
    )

    batch = make_batch(jax.random.PRNGKey(1), batch_size=BATCH_SIZE)

print("Agent created.")

2026-04-13 21:44:01.358246: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:02.657192: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:03.602123: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:04.556333: W external/xla/xla/service/gpu/au

Agent created.


In [4]:
with jax.default_device(PROFILE_DEVICE):
    # Warmup is critical on GPU because XLA autotuning may take a few iterations.
    warmup_steps = 8
    for _ in range(warmup_steps):
        agent_run, info = agent.update(batch)
    _ = info["critic/critic_loss"].block_until_ready()

    # 1) Update: post-warmup steady-state
    n_steps = 100
    t0 = time.perf_counter()
    for _ in range(n_steps):
        agent_run, info = agent_run.update(batch)
    _ = info["critic/critic_loss"].block_until_ready()
    t_steady_update = (time.perf_counter() - t0) / n_steps

    # 2) sample_actions warmup
    obs = batch["observations"]
    goals = batch["value_goals"]
    for i in range(8):
        acts = agent_run.sample_actions(obs, goals, seed=jax.random.PRNGKey(900 + i))
    _ = acts.block_until_ready()

    # 3) sample_actions steady-state
    n_steps_sample = 200
    t0 = time.perf_counter()
    for i in range(n_steps_sample):
        acts = agent_run.sample_actions(obs, goals, seed=jax.random.PRNGKey(1000 + i))
    _ = acts.block_until_ready()
    t_steady_sample = (time.perf_counter() - t0) / n_steps_sample

print("warmup update steps:", warmup_steps)
print("steady update / step:", f"{t_steady_update*1000:.3f} ms")
print("steady sample / call:", f"{t_steady_sample*1000:.3f} ms")

warmup update steps: 8
steady update / step: 89.952 ms
steady sample / call: 0.475 ms


In [5]:
def benchmark_config(overrides, n_update_steps=100, warmup_steps=8, batch_size=BATCH_SIZE):
    cfg_local = get_config()
    cfg_local.token_input_representation = "one_hot_flat"
    for k, v in overrides.items():
        setattr(cfg_local, k, v)

    with jax.default_device(PROFILE_DEVICE):
        ex_obs_local = make_one_hot_flat(jax.random.PRNGKey(11), batch_size=1)
        ex_actions_local = jnp.array([ACTION_DIM - 1], dtype=jnp.int32)
        agent_local = GCDQNTransformerAgent.create(
            seed=0,
            ex_observations=ex_obs_local,
            ex_actions=ex_actions_local,
            config=cfg_local,
        )
        batch_local = make_batch(jax.random.PRNGKey(12), batch_size=batch_size)

        for _ in range(warmup_steps):
            agent_local, info_local = agent_local.update(batch_local)
        _ = info_local["critic/critic_loss"].block_until_ready()

        t0 = time.perf_counter()
        for _ in range(n_update_steps):
            agent_local, info_local = agent_local.update(batch_local)
        _ = info_local["critic/critic_loss"].block_until_ready()
        steady = (time.perf_counter() - t0) / n_update_steps

    return {
        "steady_update_ms": steady * 1000.0,
        "overrides": overrides,
    }

configs = {
    "baseline": {
        "transformer_use_goals": True,
        "transformer_d_model": 128,
        "transformer_num_layers": 2,
        "transformer_num_heads": 4,
        "transformer_mlp_dim": 256,
        "critic_ensemble_size": 2,
    },
    "small": {
        "transformer_use_goals": False,
        "transformer_d_model": 128,
        "transformer_num_layers": 1,
        "transformer_num_heads": 4,
        "transformer_mlp_dim": 256,
        "critic_ensemble_size": 1,
    },
    "tiny": {
        "transformer_use_goals": False,
        "transformer_d_model": 64,
        "transformer_num_layers": 1,
        "transformer_num_heads": 2,
        "transformer_mlp_dim": 128,
        "critic_ensemble_size": 1,
    },
}

results = {}
for name, overrides in configs.items():
    print(f"Benchmarking {name}...")
    results[name] = benchmark_config(overrides, n_update_steps=100, warmup_steps=8, batch_size=BATCH_SIZE)

pprint(results)

Benchmarking baseline...


2026-04-13 21:44:26.819453: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:27.580087: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:28.536409: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:44:29.649154: W external/xla/xla/service/gpu/au

Benchmarking small...
Benchmarking tiny...


2026-04-13 21:45:19.542550: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:45:20.256877: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:45:21.133277: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-13 21:45:22.087708: W external/xla/xla/service/gpu/au

{'baseline': {'overrides': {'critic_ensemble_size': 2,
                            'transformer_d_model': 128,
                            'transformer_mlp_dim': 256,
                            'transformer_num_heads': 4,
                            'transformer_num_layers': 2,
                            'transformer_use_goals': True},
              'steady_update_ms': 2.721790987998247},
 'small': {'overrides': {'critic_ensemble_size': 1,
                         'transformer_d_model': 128,
                         'transformer_mlp_dim': 256,
                         'transformer_num_heads': 4,
                         'transformer_num_layers': 1,
                         'transformer_use_goals': False},
           'steady_update_ms': 0.9615339012816548},
 'tiny': {'overrides': {'critic_ensemble_size': 1,
                        'transformer_d_model': 64,
                        'transformer_mlp_dim': 128,
                        'transformer_num_heads': 2,
                        '

In [6]:
# Optional: emit a Chrome trace/TensorBoard trace for kernel-level details.
trace_dir = ROOT / "artifacts" / "jax_profile_dqn_transformer"
trace_dir.mkdir(parents=True, exist_ok=True)

agent_trace = agent_run
try:
    with jax.default_device(PROFILE_DEVICE):
        jax.profiler.start_trace(str(trace_dir))
        for _ in range(40):
            agent_trace, info_trace = agent_trace.update(batch)
        _ = info_trace["critic/critic_loss"].block_until_ready()
        jax.profiler.stop_trace()
    print("Trace written to:", trace_dir)
    print("Open with TensorBoard profile plugin, or Chrome trace tooling.")
except Exception as exc:
    print("Could not write JAX trace in this environment:", exc)

2026-04-13 21:45:40.651693: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-04-13 21:45:40.779681: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


Trace written to: /home/mbortkie/repos/golden-standard/artifacts/jax_profile_dqn_transformer
Open with TensorBoard profile plugin, or Chrome trace tooling.
